# Candidate Models

### Random Forest
An ensemble of decision trees trained with bagging. It was chosen because it:
- captures nonlinear effects and feature interactions without manual feature engineering,
- needs little tuning and is hard to overfit badly,
- provides feature importances that help interpret which factors drive turnout,
- is a different type of ensemble (bagging) from XGBoost (boosting), which gives an independent comparison.

### XGBoost
A gradient boosting model that builds trees sequentially, with each tree correcting the errors of the previous ones. It is typically the strongest performer on tabular data of this kind, and it handles interactions such as the effect of `elo_diff` depending on `log_capacity` or `day_type`. LightGBM was not trained as well, because it is another gradient boosting implementation and would be expected to perform within noise of XGBoost on a dataset of this size, adding little new information.

### Support Vector Machine (RBF kernel)
Added for diversity. Logistic regression is linear and Random Forest and XGBoost are tree-based, while an RBF SVM learns smooth, curved decision boundaries in the scaled feature space. Models that make different kinds of errors combine better in an ensemble, which is the main reason for including it. It is trained inside a pipeline with a scaler, and `probability=True` is set so that it can output class probabilities for the ensemble.

## Ensemble models

### Soft voting
Averages the predicted class probabilities of logistic regression, Random Forest, XGBoost, and SVM. It is simple, has no extra parameters to fit, and is unlikely to overfit, which makes it a reasonable first ensemble.

### Stacking
Trains a logistic regression on top of the out-of-fold predictions of the base models, so it learns how much to trust each one. It can outperform voting but is more prone to overfitting and slower to train. Because the data is split by time, the internal cross-validation uses season-aware folds instead of random folds, so that later seasons do not leak into earlier ones.


In [1]:
from pathlib import Path  
import joblib
import json  
import pandas as pd  
import matplotlib.pyplot as plt
import os

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

## Directories

In [9]:
DATA_DIR = Path("../data")
TRAIN_TEST_DIR = Path(DATA_DIR / "train_test_df")
RESULT_DIR = Path("../results")
FEATURES_DIR = Path("../data/processed/features")

## Load the train test splits and verify if they are loaded properly

In [11]:
# load the saved split
X_train = pd.read_csv(TRAIN_TEST_DIR / "X_train.csv")
X_test = pd.read_csv(TRAIN_TEST_DIR / "X_test.csv")

# select the target column so y is a Series, not a DataFrame
y_train = pd.read_csv(TRAIN_TEST_DIR / "y_train.csv")["target"]
y_test = pd.read_csv(TRAIN_TEST_DIR / "y_test.csv")["target"]

# load the feature config and class labels
with open(FEATURES_DIR / "feature_config.json") as f:
    config = json.load(f)

CLASS_LABELS = {
    int(code): name
    for code, name in config["class_labels"].items()
}

# shapes should match the previous notebook
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)

# feature names and order should match the saved config
assert X_train.columns.tolist() == config["output_columns"]
assert X_test.columns.tolist() == X_train.columns.tolist()

# rows must line up between features and targets
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# no missing values
assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
assert y_train.notna().all() and y_test.notna().all()

# classes should be the same as in the logreg notebook
print("classes:", sorted(y_train.unique()))

# class counts, which should match the earlier output
class_counts = pd.DataFrame({
    "train_count": y_train.value_counts(),
    "test_count": y_test.value_counts(),
}).sort_index()
class_counts.index = class_counts.index.map(CLASS_LABELS)
class_counts.index.name = "turnout_class"

display(class_counts)

X_train: (1559, 32) | y_train: (1559,)
X_test:  (806, 32) | y_test:  (806,)
classes: [np.int64(0), np.int64(1), np.int64(2)]


,train_count,test_count
turnout_class,,
low,520,238
medium,519,311
high,520,257


## Define the models

In [ ]:
# RBF SVM, needs scaling and probabilities for the ensemble
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, random_state=42)),
])

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
)
